RUNTIME: H100 or A100 recommended (T4 fallback)
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → H100 (or A100/T4)
2. Run this notebook from inside the cloned repo (e.g., `/content/<repo>/notebooks`) so repo data/utils resolve correctly
3. Ensure required repo data exists in `data/` (test split + any comparison splits needed)
4. Use repo paths (`data/`, `results/`) and save intermediate outputs frequently for crash safety
5. Zero-shot has no training checkpoints, so persist result CSVs after each major run block
6. Save the notebook state regularly (Ctrl+S)


In [ ]:
# ── 1. Optional Google Drive mount ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

# ── 2. Standard imports ───────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import pipeline

# ── 3. Resolve repo/data/results paths ────────────────────────────
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 4. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)
